In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# modOptDriver

modOptDriver wraps the optimizers available through the modOpt optimization framework. modOpt provides access to a variety of gradient-based and gradient-free optimization algorithms, including SLSQP, IPOPT, SNOPT, COBYLA, and others. Some of these are available in the package, some need to be separately installed, and some are commercial products that must be obtained from their respective authors (e.g. SNOPT). The modOptDriver supports sparse specification of constraint Jacobians, but not all optimizers can utilize the sparsity.

```{note}
The modOpt package does not come included with the OpenMDAO installation. It is a separate optional package that can be obtained from [LSDOlab](https://github.com/LSDOlab/modopt).
```

In this example, we use the SLSQP optimizer to minimize the objective of the Paraboloid problem.

In [ ]:
from openmdao.utils.notebook_utils import get_code
from myst_nb import glue
glue("code_src020", get_code("openmdao.test_suite.components.paraboloid.Paraboloid"), display=False)

:::{Admonition} `Paraboloid` class definition 
:class: dropdown

{glue:}`code_src020`
:::

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.modOptDriver()
prob.driver.options['optimizer'] = 'SLSQP'
prob.driver.options['maxiter'] = 200
prob.driver.options['disp'] = True

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()

prob.set_val('x', 50.0)
prob.set_val('y', 50.0)

prob.run_driver()

The "optimizer" option lets you choose which optimizer to use. modOptDriver provides access to a wide range of optimizers through the ModOpt framework. The supported optimizers are (case sensitive):

**Gradient-Based:**
- "SLSQP" - Sequential Least Squares Programming (default optimizer)
- "PySLSQP" - Pure Python implementation of SLSQP with expanded features
- "BFGS" - Broyden-Fletcher-Goldfarb-Shanno (unconstrained only)
- "LBFGSB" - Limited-memory BFGS with bounds
- "TrustConstr" - Trust-region constrained algorithm
- "IPOPT" - Interior Point Optimizer
- "SNOPT" - Sparse Nonlinear Optimizer (requires license)
- "OpenSQP" - Sequential Quadratic Programming built into ModOpt

**Gradient-Free:**
- "COBYLA" - Constrained Optimization BY Linear Approximation
- "COBYQA" - Constrained Optimization BY Quadratic Approximation (improved COBYLA)
- "NelderMead" - Nelder-Mead simplex algorithm (unconstrained only)

In [ ]:
print(prob.get_val('x'))

In [ ]:
print(prob.get_val('y'))

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal
assert_near_equal(prob.get_val('x'), 6.66666667, 1e-6)
assert_near_equal(prob.get_val('y'), -7.3333333, 1e-6)

## modOptDriver Options

In [ ]:
om.show_options_table("openmdao.drivers.modopt_driver.modOptDriver")

## modOptDriver Constructor

The call signature for the *modOptDriver* constructor is:

```{eval-rst}
    .. automethod:: openmdao.drivers.modopt_driver.modOptDriver.__init__
        :noindex:
```

## Using modOptDriver

modOptDriver has a small number of unified options that can be specified as keyword arguments when it is instantiated or by using the “options” dictionary. We have already shown how to set the optimizer option. Next we will see how `maxiter` and `disp` can be used to control the maxinum number of iterations and what the optimizer displays. The "disp" option controls optimizer output verbosity. You can set it to True/False or an integer for fine control (0=quiet, higher=more verbose). This setting is automatically mapped to the optimizer's output settings.

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.modOptDriver()
prob.driver.options['maxiter'] = 20
prob.driver.options['disp'] = False

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()

prob.set_val('x', 50.0)
prob.set_val('y', 50.0)

prob.run_driver()

In [ ]:
print(prob.get_val('x'))

In [ ]:
print(prob.get_val('y'))

In [ ]:
assert_near_equal(prob.get_val('x'), 6.66666667, 1e-6)
assert_near_equal(prob.get_val('y'), -7.3333333, 1e-6)

## modOptDriver Optimizer-Specific Options

Each optimizer in modOpt has its own set of specific options that control its behavior. You can set these optimizer-specific options through the `opt_settings` dictionary. See the [modOpt documentation](https://modopt.readthedocs.io) for detailed information about available settings for each optimizer.

For example, here's code setting some specific options for the SLSQP optimizer:

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = driver = om.modOptDriver()
driver.options['optimizer'] = 'SLSQP'
driver.options['maxiter'] = 100
driver.options['disp'] = False

# Set optimizer-specific options through opt_settings. Here we set the precision for
# the final solution with the "ftol" setting specific to SLSQP
driver.opt_settings['ftol'] = 1e-8

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()

prob.set_val('x', 50.0)
prob.set_val('y', 50.0)

prob.run_driver()

In [ ]:
assert_near_equal(prob.get_val('x'), 6.66666667, 1e-6)
assert_near_equal(prob.get_val('y'), -7.3333333, 1e-6)